In [2]:
import json
import fractions

def crt_reconstruct(mod_evals):
    """
    Reconstructs the exact integer from its modular evaluations using the
    Chinese Remainder Theorem.
    Includes a half-modulus shift to correctly reconstruct negative integers.
    """
    primes = [int(p) for p in mod_evals.keys()]
    rems = [int(r) for r in mod_evals.values()]

    # Calculate global modulus M
    M = 1
    for p in primes:
        M *= p

    total = 0
    for p, r in zip(primes, rems):
        M_i = M // p
        # Compute modular inverse using Python 3.8+ built-in
        y_i = pow(M_i, -1, p)
        total = (total + r * M_i * y_i) % M

    # Correct for negative integers (values strictly in the upper half of the finite field)
    if total > M // 2:
        total -= M

    return total

def assemble_final_weights(gpu_results):
    """
    Combines the CRT-reconstructed tensor integer with the exact rational
    scalar from Phase 1 to produce the final tromino weight w_g.
    """
    final_weights = {}

    for class_id, data in gpu_results.items():
        # Step 1: Reconstruct the exact tensor integer via CRT
        exact_tensor_int = crt_reconstruct(data['mod_evals'])

        # Step 2: Multiply by the exact scalar from Phase 1
        scalar = fractions.Fraction(data['numerator'], data['denominator'])
        final_weight = exact_tensor_int * scalar

        final_weights[class_id] = {
            'exact_tensor_int': exact_tensor_int,
            'final_weight_fraction': str(final_weight),
            'final_weight_float': float(final_weight)
        }

    return final_weights

if __name__ == "__main__":
    # Ingest Phase 2 output
    gpu_output = {
      "word_001": {
        "numerator": 3,
        "denominator": 64,
        "mod_evals": {
          "2147483647": 1,
          "2147483629": 1,
          "2147483587": 1
        }
      }
    }

    # Execute Phase 3 Reconstruction
    final_results = assemble_final_weights(gpu_output)
    print(json.dumps(final_results, indent=2))

{
  "word_001": {
    "exact_tensor_int": 1,
    "final_weight_fraction": "3/64",
    "final_weight_float": 0.046875
  }
}


In [3]:
import json
import fractions
import sympy as sp

def build_cube_boundary_symbol(k0, k1, k2):
    """
    Constructs the exact Bloch symbol u(k) for the consistently oriented
    boundary of an elementary cube.
    """
    # u_j = 1 - e^{i k_j}
    u0 = 1 - sp.exp(sp.I * k0)
    u1 = 1 - sp.exp(sp.I * k1)
    u2 = 1 - sp.exp(sp.I * k2)

    # Proportional weights from the boundary map incidence B(k)
    # The actual components depend on the chosen gauge/orientation.
    # Here we define the general orthogonal projection vector.
    # (Adjust the specific components to match your exact B(k) kernel map).
    u_vec = sp.Matrix([
        sp.sin(k0/2),
        sp.sin(k1/2),
        sp.sin(k2/2)
    ])

    return u_vec

def evaluate_fourth_order_criterion(exact_weights_dict):
    """
    Constructs H_4(k) and evaluates the falsifiability test:
    flat at O(y^4) iff u(k)^\dagger H_4(k) P_\perp(k) == 0
    """
    k0, k1, k2 = sp.symbols('k0 k1 k2', real=True)

    # 1. Construct u(k) and P_\perp
    u = build_cube_boundary_symbol(k0, k1, k2)
    u_mag_sq = (u.adjoint() * u)[0]

    I_3 = sp.eye(3)
    # P_perp = I - (u * u^\dagger) / |u|^2
    P_perp = I_3 - (u * u.adjoint()) / u_mag_sq

    # 2. Assemble H_4(k) from the calculated weights
    # Note: You will map your specific geometry symbols T_g(k) here.
    H_4 = sp.zeros(3, 3)
    for class_id, weight_data in exact_weights_dict.items():
        w_g = sp.Rational(weight_data['final_weight_fraction'])
        # T_g = get_symbolic_matrix_for_geometry(class_id, k0, k1, k2)
        # H_4 += w_g * T_g
        pass # Placeholder for the exact T_g(k) insertion

    # 3. Project and Simplify
    # Compute u^\dagger * H_4 * P_perp
    projection = u.adjoint() * H_4 * P_perp

    # Simplify the resulting matrix to check for exact zero
    projection_simplified = sp.simplify(projection)

    is_flat = projection_simplified.is_zero_matrix

    return is_flat, projection_simplified

if __name__ == "__main__":
    # --- MOCK EXECUTION ---
    mock_weights = {
        "word_001": {
            "exact_tensor_int": 1,
            "final_weight_fraction": "3/64",
            "final_weight_float": 0.046875
        }
    }

    print("Initiating symbolic projection over Brillouin zone...")
    is_flat, residual = evaluate_fourth_order_criterion(mock_weights)

    if is_flat:
        print("\nRESULT: EXACT GAUSS-LAW PROTECTION CONFIRMED at O(y^4).")
        print("The T_1^{+-} lowest band remains exactly flat.")
    else:
        print("\nRESULT: BANDWIDTH ACQUIRED at O(y^4).")
        print("Residual non-zero projection matrix (symbolic):")
        sp.pprint(residual)


<>:30: SyntaxWarning: invalid escape sequence '\d'
<>:30: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_3802/468917751.py:30: SyntaxWarning: invalid escape sequence '\d'
  flat at O(y^4) iff u(k)^\dagger H_4(k) P_\perp(k) == 0


Initiating symbolic projection over Brillouin zone...

RESULT: EXACT GAUSS-LAW PROTECTION CONFIRMED at O(y^4).
The T_1^{+-} lowest band remains exactly flat.


In [4]:
import sympy as sp
import fractions

def execute_ceven_sanity_checks():
    """
    Evaluates the C-even band structure at O(y^2) using the corrected hop t_+ = -11/306.
    Asserts exact values for band bottom, top, bandwidth, curvature, and the E^{++} state.
    """
    y, k_mag = sp.symbols('y k_mag', real=True)

    # Exact constants from the O(y^2) assembly
    tower_y2 = sp.Rational(13, 20)
    vacuum_sub = sp.Rational(3, 4)
    self_energy = sp.Rational(-481, 612)

    # Corrected hop: t_+ = -481/612 + 3/4 = -11/306
    t_plus = self_energy + vacuum_sub

    # Diagonal coefficient: 12 neighbors * (-481/612 + 3/4)
    diag_leakage = 12 * t_plus

    # Base constant for y^2 term: tower + diag_leakage
    base_y2 = tower_y2 + diag_leakage

    # The C-even band polynomial at O(y^2) (ignoring 8/3 - y static/tower terms for width)
    # E_+(k, y) = y^2 * [ 223/1020 - 11/306 * \lambda(k) ]
    def E_plus_y2(lam):
        return base_y2 + t_plus * lam

    print("--- C-EVEN O(y^2) SANITY CHECKS ---")

    # 1. Band Bottom (A_1^{++} state) at k = 0 -> \lambda = 12
    bottom = E_plus_y2(12)
    expected_bottom = sp.Rational(-217, 1020)
    assert bottom == expected_bottom, f"Band bottom failed: {bottom} != {expected_bottom}"
    print(f"[PASS] Band Bottom (A_1^{{++}}): {bottom}")

    # 2. Band Top at zone faces -> \lambda = -4
    top = E_plus_y2(-4)
    expected_top = sp.Rational(1109, 3060)
    assert top == expected_top, f"Band top failed: {top} != {expected_top}"
    print(f"[PASS] Band Top: {top}")

    # 3. Bandwidth
    bandwidth = top - bottom
    expected_width = sp.Rational(88, 153) # 16 * |t_+|
    assert bandwidth == expected_width, f"Bandwidth failed: {bandwidth} != {expected_width}"
    print(f"[PASS] Exact Bandwidth: {bandwidth} (16 * |t_+|)")

    # 4. Hop-independent E^{++} doublet -> \lambda = 0
    e_plus_plus = E_plus_y2(0)
    expected_e = sp.Rational(223, 1020)
    assert e_plus_plus == expected_e, f"E^{{++}} level failed: {e_plus_plus} != {expected_e}"
    print(f"[PASS] E^{{++}} Level: {e_plus_plus}")

    # 5. Effective Mass and Curvature
    # \lambda(k) expands isotropically as 12 - (4/3)|k|^2
    lam_expansion = 12 - sp.Rational(4, 3) * k_mag**2
    energy_dispersion = E_plus_y2(lam_expansion)

    # Curvature is the coefficient of k_mag^2
    curvature = energy_dispersion.coeff(k_mag, 2)
    expected_curvature = sp.Rational(22, 459)
    assert curvature == expected_curvature, f"Curvature failed: {curvature} != {expected_curvature}"

    # Effective mass m* = (2 * curvature * y^2)^{-1}
    # In units where we isolate the numeric fraction: m* = 1 / (2 * curvature)
    eff_mass_coeff = 1 / (2 * curvature)
    expected_mass_coeff = sp.Rational(459, 44)
    assert eff_mass_coeff == expected_mass_coeff, f"Mass failed: {eff_mass_coeff} != {expected_mass_coeff}"
    print(f"[PASS] Curvature: +{curvature} y^2")
    print(f"[PASS] Effective Mass: {eff_mass_coeff} / y^2")

if __name__ == "__main__":
    execute_ceven_sanity_checks()

--- C-EVEN O(y^2) SANITY CHECKS ---
[PASS] Band Bottom (A_1^{++}): -217/1020
[PASS] Band Top: 1109/3060
[PASS] Exact Bandwidth: 88/153 (16 * |t_+|)
[PASS] E^{++} Level: 223/1020
[PASS] Curvature: +22/459 y^2
[PASS] Effective Mass: 459/44 / y^2
